In [1]:
import pandas as pd

In [1]:
# Build merged data from DB_bank
from pathlib import Path

cwd = Path.cwd()
if (cwd / "DB_bank").exists():
    base = cwd / "DB_bank"
elif (cwd.parent / "DB_bank").exists():
    base = cwd.parent / "DB_bank"
else:
    raise FileNotFoundError("DB_bank not found. Update the base path.")

customers = pd.read_csv(base / "customers.csv")
accounts = pd.read_csv(base / "accounts.csv")
transactions = pd.read_csv(base / "transactions.csv")
str_df = pd.read_csv(base / "str.csv")

# Parse dates and numeric fields
customers["dob_or_incorporation_date"] = pd.to_datetime(
    customers["dob_or_incorporation_date"], errors="coerce"
)

for col in ["account_open_date", "last_dormant_date", "account_close_date"]:
    accounts[col] = pd.to_datetime(accounts[col], errors="coerce")

transactions["transaction_datetime"] = pd.to_datetime(
    transactions["transaction_datetime"], errors="coerce"
)
transactions["transaction_amount"] = pd.to_numeric(
    transactions["transaction_amount"], errors="coerce"
)

str_df["str_filed_date"] = pd.to_datetime(str_df["str_filed_date"], errors="coerce")

# Ensure only one STR per account_id (keep earliest)
str_df = (
    str_df
    .sort_values(["account_id", "str_filed_date"])
    .drop_duplicates(subset=["account_id"], keep="first")
)

# Merge: transactions -> accounts -> customers -> str
accounts_core = accounts.drop(columns=["customer_id"])
new_merged_data = (
    transactions
    .merge(accounts_core, on="account_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(str_df, on="account_id", how="left")
)

new_merged_data["str_filed_flag"] = new_merged_data["str_filed_date"].notna().astype("int8")


FileNotFoundError: DB_bank not found. Update the base path.

In [3]:
new_merged_data

,transaction_id,account_id,customer_id,transaction_datetime,transaction_type,transaction_category,transaction_amount,account_type,account_open_date,account_status,...,income_bracket,kyc_status,pep_flag,sanction_flag,internal_watchlist_flag,customer_risk_rating,dob_or_incorporation_date,customer_segment,str_filed_date,str_filed_flag
0,TXN_00000000,ACC_002137,CUST_01705,2024-07-07,DEBIT,RTGS,697775.69,SAVINGS,1965-05-07,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,LOW,1980-12-25,RETAIL,NaT,0
1,TXN_00000001,ACC_001962,CUST_01561,2024-12-21,DEBIT,CHEQUE,2109666.91,SAVINGS,1960-03-19,ACTIVE,...,5-10L,KYC_COMPLETE,N,N,N,MEDIUM,1960-06-16,RETAIL,NaT,0
2,TXN_00000002,ACC_000579,CUST_00467,2024-07-24,DEBIT,CHEQUE,994268.37,SAVINGS,2010-09-28,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,LOW,2003-09-25,RETAIL,NaT,0
3,TXN_00000003,ACC_008046,CUST_06436,2024-08-26,CREDIT,FX,4346514.67,SAVINGS,1991-12-05,ACTIVE,...,10-25L,KYC_COMPLETE,N,N,N,LOW,1965-11-14,RETAIL,NaT,0
4,TXN_00000004,ACC_003043,CUST_02444,2024-12-26,CREDIT,RTGS,1076647.28,SAVINGS,2006-07-14,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,LOW,1967-12-21,RETAIL,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,TXN_00249995,ACC_002039,CUST_01620,2024-11-01,CREDIT,IMPS,318211.98,SAVINGS,2004-01-09,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,MEDIUM,1938-07-24,RETAIL,NaT,0
249996,TXN_00249996,ACC_005537,CUST_04439,2024-07-14,CREDIT,NEFT,3109135.17,SAVINGS,1965-04-28,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,HIGH,1951-06-01,RETAIL,2024-11-07,1
249997,TXN_00249997,ACC_004268,CUST_03413,2024-07-06,CREDIT,NEFT,2638231.05,SAVINGS,1976-09-18,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,LOW,1958-08-09,RETAIL,NaT,0
249998,TXN_00249998,ACC_004998,CUST_03994,2024-10-17,CREDIT,NEFT,3930338.42,SAVINGS,1963-03-19,ACTIVE,...,<5L,KYC_COMPLETE,N,N,N,HIGH,1973-05-07,RETAIL,NaT,0


In [7]:
new_merged_data['transaction_datetime'] = pd.to_datetime(
    new_merged_data['transaction_datetime'],
    errors='coerce'
)


# -------------------------------------------------------------------
# STEP 0 – sanity check
# -------------------------------------------------------------------

In [11]:
step0 = (
    new_merged_data
    .loc[
        (new_merged_data['account_id'] == 'ACC_000001') &
        (new_merged_data['transaction_type'] == 'DEBIT')
    ,
        ['transaction_datetime', 'transaction_amount']
    ]
    .sort_values('transaction_datetime')
)

print("STEP 0 - sanity check (ACC_000001, DEBIT):")
step0.head(20)

STEP 0 - sanity check (ACC_000001, DEBIT):


,transaction_datetime,transaction_amount
236809,2024-07-01,44281.22
93156,2024-09-12,1210456.35
183660,2024-09-21,26207.81
113122,2024-09-27,2165984.21
183468,2024-12-07,34297.13


# -------------------------------------------------------------------
# STEP 1 – static filter: only DEBIT transactions
# -------------------------------------------------------------------

In [13]:
step1_monthly = new_merged_data[new_merged_data['transaction_type'] == 'DEBIT'].copy()

print("\nSTEP 1 - row count (DEBIT only):", len(step1_monthly))
print("\nSTEP 1 - sample:")
step1_monthly.sort_values(['account_id', 'transaction_datetime']).head(5)



STEP 1 - row count (DEBIT only): 125115

STEP 1 - sample:


,transaction_id,account_id,customer_id,transaction_datetime,transaction_type,transaction_category,transaction_amount,accounts.account_id,accounts.customer_id,accounts.account_type,...,customers.income_bracket,customers.kyc_status,customers.pep_flag,customers.sanction_flag,customers.internal_watchlist_flag,customers.customer_risk_rating,customers.dob_or_incorporation_date,customers.customer_segment,str.account_id,str.str_filed_date
236809,TXN_00235831,ACC_000001,CUST_00001,2024-07-01,DEBIT,CASH,44281.22,ACC_000001,CUST_00001,SAVINGS,...,<5L,KYC_COMPLETE,N,N,N,LOW,09/03/1950,RETAIL,NaN,NaN
93156,TXN_00092771,ACC_000001,CUST_00001,2024-09-12,DEBIT,FX,1210456.35,ACC_000001,CUST_00001,SAVINGS,...,<5L,KYC_COMPLETE,N,N,N,LOW,09/03/1950,RETAIL,NaN,NaN
183660,TXN_00182883,ACC_000001,CUST_00001,2024-09-21,DEBIT,CASH,26207.81,ACC_000001,CUST_00001,SAVINGS,...,<5L,KYC_COMPLETE,N,N,N,LOW,09/03/1950,RETAIL,NaN,NaN
113122,TXN_00112646,ACC_000001,CUST_00001,2024-09-27,DEBIT,INTERNATIONAL_WIRE,2165984.21,ACC_000001,CUST_00001,SAVINGS,...,<5L,KYC_COMPLETE,N,N,N,LOW,09/03/1950,RETAIL,NaN,NaN
183468,TXN_00182691,ACC_000001,CUST_00001,2024-12-07,DEBIT,CASH,34297.13,ACC_000001,CUST_00001,SAVINGS,...,<5L,KYC_COMPLETE,N,N,N,LOW,09/03/1950,RETAIL,NaN,NaN


# -------------------------------------------------------------------
# STEP 1.5 – grouping at daily level
# (Here you group by the full transaction_datetime in SQL; we mimic that)
# -------------------------------------------------------------------

In [27]:
step_2_5_daily_level = (
    step1_monthly
    .assign(
        month_last_date = step1_monthly['transaction_datetime'].dt.to_period('M').dt.to_timestamp('M') + pd.Timedelta(days=1)
        # SQL: DATEADD(day, 1, EOMONTH(transaction_datetime))
    )
    .groupby(['account_id', 'customer_id', 'transaction_datetime', 'month_last_date'], as_index=False)
    .agg(total_daily_amount=('transaction_amount', 'sum'))
)

print("\nSTEP 1.5 - daily level for ACC_000001:")
step_2_5_daily_level.loc[step_2_5_daily_level['account_id'] == 'ACC_000001'].sort_values('transaction_datetime')


STEP 1.5 - daily level for ACC_000001:


,account_id,customer_id,transaction_datetime,month_last_date,total_daily_amount
0,ACC_000001,CUST_00001,2024-07-01,2024-08-01,44281.22
1,ACC_000001,CUST_00001,2024-09-12,2024-10-01,1210456.35
2,ACC_000001,CUST_00001,2024-09-21,2024-10-01,26207.81
3,ACC_000001,CUST_00001,2024-09-27,2024-10-01,2165984.21
4,ACC_000001,CUST_00001,2024-12-07,2025-01-01,34297.13


# -------------------------------------------------------------------
# STEP 2.1 – lookback on 7 days (daily - transaction_datetime column)
# Left join step_2_5_daily_level (a) with step_2_5_daily_level (b)
# where account_id matches and b.transaction_datetime between
# a.transaction_datetime - 7 days and a.transaction_datetime
# -------------------------------------------------------------------

In [16]:
a = step_2_5_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_a',
    'total_daily_amount': 'total_daily_amount_a'
})

In [18]:
b = step_2_5_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_b',
    'total_daily_amount': 'total_daily_amount_b'
})

In [20]:
merged_ab = a.merge(
    b,
    on='account_id',
    suffixes=('_a', '_b'),
    how='left'
)

# Filter on the 7-day lookback window
mask_7d = (
    (merged_ab['transaction_datetime_b'] >= merged_ab['transaction_datetime_a'] - pd.Timedelta(days=7)) &
    (merged_ab['transaction_datetime_b'] <= merged_ab['transaction_datetime_a'])
)

step3_lookback_table = (
    merged_ab.loc[mask_7d, [
        'account_id',
        'customer_id_a',
        'transaction_datetime_a',
        'total_daily_amount_a',
        'total_daily_amount_b',
        'transaction_datetime_b'
    ]]
    .rename(columns={
        'customer_id_a': 'customer_id',
        'transaction_datetime_a': 'transaction_datetime',
        'total_daily_amount_a': 'total_daily_amount',
        'total_daily_amount_b': 'amount_lookback',
        'transaction_datetime_b': 'trxn_date_daily_level'
    })
    .sort_values(['account_id', 'transaction_datetime', 'trxn_date_daily_level'])
)

print("\nSTEP 2.1 - 7-day lookback for ACC_000001:")
step3_lookback_table.loc[step3_lookback_table['account_id'] == 'ACC_000001'].head(50)



STEP 2.1 - 7-day lookback for ACC_000001:


,account_id,customer_id,transaction_datetime,total_daily_amount,amount_lookback,trxn_date_daily_level
0,ACC_000001,CUST_00001,2024-07-01,44281.22,44281.22,2024-07-01
6,ACC_000001,CUST_00001,2024-09-12,1210456.35,1210456.35,2024-09-12
12,ACC_000001,CUST_00001,2024-09-21,26207.81,26207.81,2024-09-21
17,ACC_000001,CUST_00001,2024-09-27,2165984.21,26207.81,2024-09-21
18,ACC_000001,CUST_00001,2024-09-27,2165984.21,2165984.21,2024-09-27
24,ACC_000001,CUST_00001,2024-12-07,34297.13,34297.13,2024-12-07


# -------------------------------------------------------------------
# STEP 2.2 – lookback on 60 days (monthly - transaction_datetime)
# Logic:
#  - For each (account_id, month_last_date), find the first row by transaction_datetime
#    (rank_1 = 1).
#  - For each row a, join to rows d where a.account_id = d.account_id and
#    a.transaction_datetime between (d.month_last_date - 60 days) and d.month_last_date.
# -------------------------------------------------------------------

In [31]:
# -------------------------------------------------------------------
# STEP 2.2 – lookback on 60 days (monthly - transaction_datetime)
# FIXED VERSION
# -------------------------------------------------------------------

# First: compute rank_1 per (account_id, month_last_date)
step_2_5_daily_level_sorted = (
    step_2_5_daily_level
    .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
)

# Get d: first row per (account_id, month_last_date)
d = (
    step_2_5_daily_level_sorted
    .drop_duplicates(subset=['account_id', 'month_last_date'], keep='first')
    [['account_id', 'customer_id', 'month_last_date']]
    .copy()
)

print("\nDEBUG - d (first row per month):")
print(d[d['account_id'] == 'ACC_000001'].head())

# Prepare a2
a2 = step_2_5_daily_level.copy()

print("\nDEBUG - a2 columns:", a2.columns.tolist())
print("\nDEBUG - a2 sample:")
print(a2[a2['account_id'] == 'ACC_000001'].head())

# Do a cross join on account_id, customer_id
# This creates month_last_date_x (from a2) and month_last_date_y (from d)
merged_a2d = a2.merge(
    d,
    on=['account_id', 'customer_id'],
    how='left',
    suffixes=('_x', '_y')
)

print("\nDEBUG - merged_a2d columns after merge:", merged_a2d.columns.tolist())
print("\nDEBUG - merged_a2d shape:", merged_a2d.shape)
print("\nDEBUG - merged_a2d sample:")
print(merged_a2d[merged_a2d['account_id'] == 'ACC_000001'].head(10))

# Apply 60-day lookback window filter
# Use month_last_date_y (from d, the reference month-end)
# and transaction_datetime (from a2, the transaction date)
mask_60d = (
    (merged_a2d['transaction_datetime'] >= merged_a2d['month_last_date_y'] - pd.Timedelta(days=60)) &
    (merged_a2d['transaction_datetime'] <= merged_a2d['month_last_date_y'])
)

print("\nDEBUG - rows passing 60d mask:", mask_60d.sum())

# Remove duplicates and create final result
step3_lookback_table_monthly = (
    merged_a2d
    .loc[mask_60d]
    .drop_duplicates(
        subset=['account_id', 'customer_id', 'transaction_datetime', 'month_last_date_y'],
        keep='first'
    )
    .rename(columns={
        'total_daily_amount': 'lookbakc_amt',
        'month_last_date_y': 'month_last_date'  # Use the reference month-end from d
    })
    [['account_id', 'customer_id', 'transaction_datetime', 'lookbakc_amt', 'month_last_date']]
    .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
    .reset_index(drop=True)
)

print("\nSTEP 2.2 - step3_lookback_table_monthly for ACC_000001:")
step3_lookback_table_monthly.loc[step3_lookback_table_monthly['account_id'] == 'ACC_000001']\
    .sort_values('month_last_date')


DEBUG - d (first row per month):
   account_id customer_id month_last_date
0  ACC_000001  CUST_00001      2024-08-01
1  ACC_000001  CUST_00001      2024-10-01
4  ACC_000001  CUST_00001      2025-01-01

DEBUG - a2 columns: ['account_id', 'customer_id', 'transaction_datetime', 'month_last_date', 'total_daily_amount']

DEBUG - a2 sample:
   account_id customer_id transaction_datetime month_last_date  \
0  ACC_000001  CUST_00001           2024-07-01      2024-08-01   
1  ACC_000001  CUST_00001           2024-09-12      2024-10-01   
2  ACC_000001  CUST_00001           2024-09-21      2024-10-01   
3  ACC_000001  CUST_00001           2024-09-27      2024-10-01   
4  ACC_000001  CUST_00001           2024-12-07      2025-01-01   

   total_daily_amount  
0            44281.22  
1          1210456.35  
2            26207.81  
3          2165984.21  
4            34297.13  

DEBUG - merged_a2d columns after merge: ['account_id', 'customer_id', 'transaction_datetime', 'month_last_date_x', 'tota

,account_id,customer_id,transaction_datetime,lookbakc_amt,month_last_date
0,ACC_000001,CUST_00001,2024-07-01,44281.22,2024-08-01
1,ACC_000001,CUST_00001,2024-09-12,1210456.35,2024-10-01
2,ACC_000001,CUST_00001,2024-09-21,26207.81,2024-10-01
3,ACC_000001,CUST_00001,2024-09-27,2165984.21,2024-10-01
4,ACC_000001,CUST_00001,2024-12-07,34297.13,2025-01-01


# -------------------------------------------------------------------
# STEP 3.1 – Daily threshold (7-day lookback)
# group by account_id, transaction_datetime, sum(amount_lookback)
# -------------------------------------------------------------------

In [32]:
daily_threshold = (
    step3_lookback_table
    .groupby(['account_id', 'transaction_datetime'], as_index=False)
    .agg(threshold_amt=('amount_lookback', 'sum'))
)

print("\nSTEP 3.1 - daily threshold for ACC_000001:")
daily_threshold.loc[daily_threshold['account_id'] == 'ACC_000001'].sort_values('transaction_datetime')


STEP 3.1 - daily threshold for ACC_000001:


,account_id,transaction_datetime,threshold_amt
0,ACC_000001,2024-07-01,44281.22
1,ACC_000001,2024-09-12,1210456.35
2,ACC_000001,2024-09-21,26207.81
3,ACC_000001,2024-09-27,2192192.02
4,ACC_000001,2024-12-07,34297.13


# -------------------------------------------------------------------
# STEP 3.2 – Monthly threshold (60-day lookback)
# group by account_id, month_last_date, sum(lookbakc_amt)
# -------------------------------------------------------------------

In [34]:
monthly_threshold = (
    step3_lookback_table_monthly
    .groupby(['account_id', 'month_last_date'], as_index=False)
    .agg(threshold_amt=('lookbakc_amt', 'sum'))
)

print("\nSTEP 3.2 - monthly threshold for ACC_000001:")
monthly_threshold.loc[monthly_threshold['account_id'] == 'ACC_000001'].sort_values('month_last_date')


STEP 3.2 - monthly threshold for ACC_000001:


,account_id,month_last_date,threshold_amt
0,ACC_000001,2024-08-01,44281.22
1,ACC_000001,2024-10-01,3402648.37
2,ACC_000001,2025-01-01,34297.13


In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

# =====================================================================
# CONFIGURATION - CHANGE THESE PARAMETERS
# =====================================================================
TRANSACTION_TYPE = 'DEBIT'  # 'DEBIT', 'CREDIT', or 'ALL'
AGGREGATION_LEVEL = 'daily'  # 'daily' or 'monthly'
LOOKBACK_DAYS = 10 # 7, 10, 14, 20, 30, 60, 90, etc.
DEBUG = True

# =====================================================================
# STEP 0 – Load and validate data
# =====================================================================
print("="*70)
print(f"CONFIG: TYPE={TRANSACTION_TYPE}, LEVEL={AGGREGATION_LEVEL}, LOOKBACK={LOOKBACK_DAYS} days")
print("="*70)

# Ensure transaction_datetime is datetime
new_merged_data['transaction_datetime'] = pd.to_datetime(new_merged_data['transaction_datetime'])

# =====================================================================
# STEP 1 – Filter by transaction type
# =====================================================================
if TRANSACTION_TYPE == 'ALL':
    step1_filtered = new_merged_data.copy()
else:
    step1_filtered = new_merged_data[new_merged_data['transaction_type'] == TRANSACTION_TYPE].copy()

print(f"\nSTEP 1 - Row count ({TRANSACTION_TYPE}):", len(step1_filtered))
print(f"Sample:")
print(step1_filtered.sort_values(['account_id', 'transaction_datetime']).head(10))

# =====================================================================
# STEP 1.5 – Group to daily/monthly level
# =====================================================================
if AGGREGATION_LEVEL == 'daily':
    # Group by account, customer, and exact transaction_datetime
    step_daily_level = (
        step1_filtered
        .assign(
            month_last_date = step1_filtered['transaction_datetime'].dt.to_period('M').dt.to_timestamp('M') + pd.Timedelta(days=1)
        )
        .groupby(['account_id', 'customer_id', 'transaction_datetime', 'month_last_date'], as_index=False)
        .agg(total_daily_amount=('transaction_amount', 'sum'))
    )
    aggregation_key = 'transaction_datetime'
    
else:  # monthly
    # Group by account, customer, and month
    step_daily_level = (
        step1_filtered
        .assign(
            month_last_date = step1_filtered['transaction_datetime'].dt.to_period('M').dt.to_timestamp('M') + pd.Timedelta(days=1),
            month_start = step1_filtered['transaction_datetime'].dt.to_period('M').dt.to_timestamp()
        )
        .groupby(['account_id', 'customer_id', 'month_start', 'month_last_date'], as_index=False)
        .agg(total_daily_amount=('transaction_amount', 'sum'))
        .rename(columns={'month_start': 'transaction_datetime'})
    )
    aggregation_key = 'transaction_datetime'

print(f"\nSTEP 1.5 - Aggregated to {AGGREGATION_LEVEL} level: {len(step_daily_level)} rows")
print(step_daily_level.head(10))

# =====================================================================
# STEP 2 – Lookback join (PARAMETERIZED)
# =====================================================================
print(f"\nSTEP 2 - Creating {LOOKBACK_DAYS}-day lookback join...")

# Prepare a (current transactions)
a = step_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_a',
    'total_daily_amount': 'total_daily_amount_a'
})

# Prepare b (historical transactions for lookback)
b = step_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_b',
    'total_daily_amount': 'total_daily_amount_b'
})

# Merge on account_id
merged_ab = a.merge(
    b,
    on='account_id',
    suffixes=('_a', '_b'),
    how='left'
)

# Filter on lookback window
mask_lookback = (
    (merged_ab['transaction_datetime_b'] >= merged_ab['transaction_datetime_a'] - pd.Timedelta(days=LOOKBACK_DAYS)) &
    (merged_ab['transaction_datetime_b'] <= merged_ab['transaction_datetime_a'])
)

lookback_table = (
    merged_ab.loc[mask_lookback, [
        'account_id',
        'customer_id_a',
        'transaction_datetime_a',
        'total_daily_amount_a',
        'total_daily_amount_b',
        'transaction_datetime_b'
    ]]
    .rename(columns={
        'customer_id_a': 'customer_id',
        'transaction_datetime_a': 'transaction_datetime',
        'total_daily_amount_a': 'total_daily_amount',
        'total_daily_amount_b': 'amount_lookback',
        'transaction_datetime_b': 'trxn_date_lookback'
    })
    .sort_values(['account_id', 'transaction_datetime', 'trxn_date_lookback'])
)

print(f"STEP 2 - Lookback rows created: {len(lookback_table)}")

# =====================================================================
# STEP 3 – Aggregation to get thresholds
# =====================================================================
threshold_table = (
    lookback_table
    .groupby(['account_id', 'customer_id', 'transaction_datetime'], as_index=False)
    .agg(
        threshold_amt=('amount_lookback', 'sum'),
        trxn_count=('amount_lookback', 'count'),
        avg_amt=('amount_lookback', 'mean'),
        max_amt=('amount_lookback', 'max'),
        min_amt=('amount_lookback', 'min')
    )
)

print(f"\nSTEP 3 - Threshold table created: {len(threshold_table)} rows")
print(threshold_table.head(20))

# =====================================================================
# WORST CASE ANALYSIS
# =====================================================================
print("\n" + "="*70)
print("WORST CASE POPULATION ANALYSIS")
print("="*70)

worst_case = threshold_table.groupby('account_id').agg(
    count_periods=('account_id', 'count'),
    total_threshold=('threshold_amt', 'sum'),
    avg_threshold=('threshold_amt', 'mean'),
    max_threshold=('threshold_amt', 'max'),
    min_threshold=('threshold_amt', 'min'),
    total_trxn_count=('trxn_count', 'sum')
).reset_index().sort_values('total_threshold', ascending=False)

print(f"\nTop 20 accounts by total threshold amount:")
print(worst_case.head(20))

print(f"\nWorst case (highest single threshold):")
worst_single = threshold_table.nlargest(10, 'threshold_amt')[['account_id', 'customer_id', 'transaction_datetime', 'threshold_amt', 'trxn_count']]
print(worst_single)

print(f"\nStatistics:")
print(f"  Total unique accounts: {threshold_table['account_id'].nunique()}")
print(f"  Total periods analyzed: {len(threshold_table)}")
print(f"  Average threshold: {threshold_table['threshold_amt'].mean():.2f}")
print(f"  Median threshold: {threshold_table['threshold_amt'].median():.2f}")
print(f"  Max threshold: {threshold_table['threshold_amt'].max():.2f}")
print(f"  Min threshold: {threshold_table['threshold_amt'].min():.2f}")
print(f"  Std deviation: {threshold_table['threshold_amt'].std():.2f}")

# =====================================================================
# CORRECTED STEP 2.2 FOR MONTHLY REFERENCE POINTS
# =====================================================================
print("\n" + "="*70)
print("STEP 2.2 - MONTHLY REFERENCE POINTS (if applicable)")
print("="*70)

if AGGREGATION_LEVEL == 'daily':
    # Create monthly reference points
    step_2_5_daily_level_sorted = (
        step_daily_level
        .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
    )
    
    # Get first row per (account_id, month_last_date)
    d = (
        step_2_5_daily_level_sorted
        .drop_duplicates(subset=['account_id', 'month_last_date'], keep='first')
        [['account_id', 'customer_id', 'month_last_date']]
        .copy()
    )
    
    if DEBUG:
        print("\nDEBUG - d (first row per month):")
        print(d.head(10))
    
    # Prepare a2 (all daily transactions)
    a2 = step_daily_level.copy()
    
    # Cross join on account_id, customer_id
    merged_a2d = a2.merge(
        d,
        on=['account_id', 'customer_id'],
        how='left',
        suffixes=('_x', '_y')
    )
    
    if DEBUG:
        print(f"\nDEBUG - merged_a2d shape before filter: {merged_a2d.shape}")
        print(f"DEBUG - merged_a2d columns: {merged_a2d.columns.tolist()}")
    
    # Apply LOOKBACK_DAYS window filter
    # Use month_last_date_y (from d, the reference month-end)
    mask_lookback_monthly = (
        (merged_a2d['transaction_datetime'] >= merged_a2d['month_last_date_y'] - pd.Timedelta(days=LOOKBACK_DAYS)) &
        (merged_a2d['transaction_datetime'] <= merged_a2d['month_last_date_y'])
    )
    
    if DEBUG:
        print(f"DEBUG - rows passing {LOOKBACK_DAYS}-day mask: {mask_lookback_monthly.sum()}")
    
    # Remove duplicates and create final result
    step3_lookback_table_monthly = (
        merged_a2d
        .loc[mask_lookback_monthly]
        .drop_duplicates(
            subset=['account_id', 'customer_id', 'transaction_datetime', 'month_last_date_y'],
            keep='first'
        )
        .rename(columns={
            'total_daily_amount': 'lookbakc_amt',
            'month_last_date_y': 'month_last_date'
        })
        [['account_id', 'customer_id', 'transaction_datetime', 'lookbakc_amt', 'month_last_date']]
        .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
        .reset_index(drop=True)
    )
    
    print(f"\nSTEP 2.2 - Monthly reference lookback: {len(step3_lookback_table_monthly)} rows")
    print(step3_lookback_table_monthly.head(20))
    
    # Monthly threshold aggregation
    monthly_threshold = (
        step3_lookback_table_monthly
        .groupby(['account_id', 'month_last_date'], as_index=False)
        .agg(
            threshold_amt=('lookbakc_amt', 'sum'),
            transaction_count=('lookbakc_amt', 'count')
        )
    )
    
    print(f"\nMonthly threshold aggregation:")
    print(monthly_threshold.head(20))

else:  # Already monthly, no need for step 2.2
    print("Note: Aggregation level is already 'monthly', Step 2.2 not applicable.")

# =====================================================================
# EXPORT RESULTS
# =====================================================================
print("\n" + "="*70)
print("EXPORTING RESULTS")
print("="*70)

export_name = f"threshold_{TRANSACTION_TYPE}_{AGGREGATION_LEVEL}_{LOOKBACK_DAYS}d_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

threshold_table.to_csv(f"{export_name}_thresholds.csv", index=False)
worst_case.to_csv(f"{export_name}_worst_case.csv", index=False)

print(f"\nFiles exported:")
print(f"  - {export_name}_thresholds.csv")
print(f"  - {export_name}_worst_case.csv")

print("\n✓ Script completed successfully!")

CONFIG: TYPE=DEBIT, LEVEL=daily, LOOKBACK=10 days

STEP 1 - Row count (DEBIT): 124570
Sample:
       transaction_id  account_id customer_id transaction_datetime  \
235831   TXN_00235831  ACC_000001  CUST_00001           2024-07-01   
92771    TXN_00092771  ACC_000001  CUST_00001           2024-09-12   
182883   TXN_00182883  ACC_000001  CUST_00001           2024-09-21   
112646   TXN_00112646  ACC_000001  CUST_00001           2024-09-27   
182691   TXN_00182691  ACC_000001  CUST_00001           2024-12-07   
124896   TXN_00124896  ACC_000002  CUST_00002           2024-07-13   
160340   TXN_00160340  ACC_000002  CUST_00002           2024-08-22   
26943    TXN_00026943  ACC_000002  CUST_00002           2024-09-02   
171058   TXN_00171058  ACC_000002  CUST_00002           2024-09-08   
220744   TXN_00220744  ACC_000002  CUST_00002           2024-09-16   

       transaction_type transaction_category  transaction_amount account_type  \
235831            DEBIT                 CASH          

In [5]:
threshold_table

,account_id,customer_id,transaction_datetime,threshold_amt,trxn_count,avg_amt,max_amt,min_amt
0,ACC_000001,CUST_00001,2024-07-01,44281.22,1,4.428122e+04,44281.22,44281.22
1,ACC_000001,CUST_00001,2024-09-12,1210456.35,1,1.210456e+06,1210456.35,1210456.35
2,ACC_000001,CUST_00001,2024-09-21,1236664.16,2,6.183321e+05,1210456.35,26207.81
3,ACC_000001,CUST_00001,2024-09-27,2192192.02,2,1.096096e+06,2165984.21,26207.81
4,ACC_000001,CUST_00001,2024-12-07,34297.13,1,3.429713e+04,34297.13,34297.13
...,...,...,...,...,...,...,...,...
119651,ACC_009979,CUST_08000,2024-12-08,2542689.40,1,2.542689e+06,2542689.40,2542689.40
119652,ACC_009979,CUST_08000,2024-12-09,2640648.40,2,1.320324e+06,2542689.40,97959.00
119653,ACC_009979,CUST_08000,2024-12-20,2508868.20,1,2.508868e+06,2508868.20,2508868.20
119654,ACC_009979,CUST_08000,2024-12-26,3756989.15,2,1.878495e+06,2508868.20,1248120.95


In [ ]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# STEP 0 – sanity check
# -------------------------------------------------------------------
step0 = (
    new_merged_data
    .loc[
        (new_merged_data['account_id'] == 'ACC_000001') &
        (new_merged_data['transaction_type'] == 'DEBIT')
    ,
        ['transaction_datetime', 'transaction_amount']
    ]
    .sort_values('transaction_datetime')
)

print("STEP 0 - sanity check (ACC_000001, DEBIT):")
print(step0.head(20))

# -------------------------------------------------------------------
# STEP 1 – static filter: only DEBIT transactions
# -------------------------------------------------------------------
step1_monthly = new_merged_data[new_merged_data['transaction_type'] == 'DEBIT'].copy()

print("\nSTEP 1 - row count (DEBIT only):", len(step1_monthly))
print("\nSTEP 1 - sample:")
print(
    step1_monthly
    .sort_values(['account_id', 'transaction_datetime'])
    .head(20)
)

# -------------------------------------------------------------------
# STEP 1.5 – grouping at daily level
# (Here you group by the full transaction_datetime in SQL; we mimic that)
# -------------------------------------------------------------------
step_2_5_daily_level = (
    step1_monthly
    .assign(
        month_last_date = step1_monthly['transaction_datetime'].dt.to_period('M').dt.to_timestamp('M') + pd.Timedelta(days=1)
        # SQL: DATEADD(day, 1, EOMONTH(transaction_datetime))
    )
    .groupby(['account_id', 'customer_id', 'transaction_datetime', 'month_last_date'], as_index=False)
    .agg(total_daily_amount=('transaction_amount', 'sum'))
)

print("\nSTEP 1.5 - daily level for ACC_000001:")
print(
    step_2_5_daily_level
    .loc[step_2_5_daily_level['account_id'] == 'ACC_000001']
    .sort_values('transaction_datetime')
)

# -------------------------------------------------------------------
# STEP 2.1 – lookback on 7 days (daily - transaction_datetime column)
# Left join step_2_5_daily_level (a) with step_2_5_daily_level (b)
# where account_id matches and b.transaction_datetime between
# a.transaction_datetime - 7 days and a.transaction_datetime
# -------------------------------------------------------------------
a = step_2_5_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_a',
    'total_daily_amount': 'total_daily_amount_a'
})
b = step_2_5_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_b',
    'total_daily_amount': 'total_daily_amount_b'
})

# Merge on account_id and customer_id (SQL uses account_id; you can decide
# whether to extend to customer_id as well; here we keep only account_id
# to match your SQL)
merged_ab = a.merge(
    b,
    on='account_id',
    suffixes=('_a', '_b'),
    how='left'
)

# Filter on the 7-day lookback window
mask_7d = (
    (merged_ab['transaction_datetime_b'] >= merged_ab['transaction_datetime_a'] - pd.Timedelta(days=7)) &
    (merged_ab['transaction_datetime_b'] <= merged_ab['transaction_datetime_a'])
)

step3_lookback_table = (
    merged_ab.loc[mask_7d, [
        'account_id',
        'customer_id_a',
        'transaction_datetime_a',
        'total_daily_amount_a',
        'total_daily_amount_b',
        'transaction_datetime_b'
    ]]
    .rename(columns={
        'customer_id_a': 'customer_id',
        'transaction_datetime_a': 'transaction_datetime',
        'total_daily_amount_a': 'total_daily_amount',
        'total_daily_amount_b': 'amount_lookback',
        'transaction_datetime_b': 'trxn_date_daily_level'
    })
    .sort_values(['account_id', 'transaction_datetime', 'trxn_date_daily_level'])
)

print("\nSTEP 2.1 - 7-day lookback for ACC_000001:")
print(
    step3_lookback_table
    .loc[step3_lookback_table['account_id'] == 'ACC_000001']
    .head(50)
)

# -------------------------------------------------------------------
# STEP 2.2 – lookback on 60 days (monthly - transaction_datetime)
# Logic:
#  - For each (account_id, month_last_date), find the first row by transaction_datetime
#    (rank_1 = 1).
#  - For each row a, join to rows d where a.account_id = d.account_id and
#    a.transaction_datetime between (d.month_last_date - 60 days) and d.month_last_date.
# -------------------------------------------------------------------

# First: compute rank_1 per (account_id, month_last_date)
step_2_5_daily_level_sorted = (
    step_2_5_daily_level
    .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
)
step_2_5_daily_level_sorted['rank_1'] = (
    step_2_5_daily_level_sorted
    .groupby(['account_id', 'month_last_date'])
    .cumcount() + 1
)

d = step_2_5_daily_level_sorted[step_2_5_daily_level_sorted['rank_1'] == 1].copy()
d = d.rename(columns={
    'transaction_datetime': 'transaction_datetime_d',
    'total_daily_amount': 'total_daily_amount_d'
})

a2 = step_2_5_daily_level.rename(columns={
    'transaction_datetime': 'transaction_datetime_a2',
    'total_daily_amount': 'total_daily_amount_a2'
})

merged_a2d = a2.merge(
    d[['account_id', 'month_last_date', 'transaction_datetime_d', 'total_daily_amount_d', 'rank_1']],
    on='account_id',
    how='left',
    suffixes=('', '_d')
)

mask_60d = (
    (merged_a2d['transaction_datetime_a2'] >= merged_a2d['month_last_date'] - pd.Timedelta(days=60)) &
    (merged_a2d['transaction_datetime_a2'] <= merged_a2d['month_last_date'])
)

step3_lookback_table_monthly = (
    merged_a2d.loc[mask_60d, [
        'account_id',
        'customer_id',
        'transaction_datetime_a2',
        'total_daily_amount_a2',
        'month_last_date',
        'rank_1'
    ]]
    .rename(columns={
        'transaction_datetime_a2': 'transaction_datetime',
        'total_daily_amount_a2': 'lookbakc_amt'  # keep the same name as in SQL
    })
    .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
)

print("\nSTEP 2.2 - rank_1 view for ACC_000001:")
print(
    step_2_5_daily_level_sorted
    .loc[step_2_5_daily_level_sorted['account_id'] == 'ACC_000001', 
         ['account_id', 'transaction_datetime', 'month_last_date', 'rank_1']]
    .sort_values(['account_id', 'month_last_date', 'transaction_datetime'])
)

print("\nSTEP 2.2 - step3_lookback_table_monthly for ACC_000001:")
print(
    step3_lookback_table_monthly
    .loc[step3_lookback_table_monthly['account_id'] == 'ACC_000001']
    .sort_values('month_last_date')
)

# -------------------------------------------------------------------
# STEP 3.1 – Daily threshold (7-day lookback)
# group by account_id, transaction_datetime, sum(amount_lookback)
# -------------------------------------------------------------------
daily_threshold = (
    step3_lookback_table
    .groupby(['account_id', 'transaction_datetime'], as_index=False)
    .agg(threshold_amt=('amount_lookback', 'sum'))
)

print("\nSTEP 3.1 - daily threshold for ACC_000001:")
print(
    daily_threshold
    .loc[daily_threshold['account_id'] == 'ACC_000001']
    .sort_values('transaction_datetime')
)

# -------------------------------------------------------------------
# STEP 3.2 – Monthly threshold (60-day lookback)
# group by account_id, month_last_date, sum(lookbakc_amt)
# -------------------------------------------------------------------
monthly_threshold = (
    step3_lookback_table_monthly
    .groupby(['account_id', 'month_last_date'], as_index=False)
    .agg(threshold_amt=('lookbakc_amt', 'sum'))
)

print("\nSTEP 3.2 - monthly threshold for ACC_000001:")
print(
    monthly_threshold
    .loc[monthly_threshold['account_id'] == 'ACC_000001']
    .sort_values('month_last_date')
)

# -------------------------------------------------------------------
# OPTIONAL: If you also want to reproduce your second SQL block
# (step2_monthly and step3_monthly) using the original "merged data"
# assuming new_merged_data is the same as [merged data] but with DEBIT filter removed.
# -------------------------------------------------------------------

# STEP 2 – 1-month lookback over all transactions (no DEBIT filter)
a_m = new_merged_data.rename(columns={
    'transaction_datetime': 'transaction_datetime_a',
    'transaction_amount': 'transaction_amount_a',
})
b_m = new_merged_data.rename(columns={
    'transaction_datetime': 'transaction_datetime_b',
    'transaction_amount': 'transaction_amount_b',
})

merged_m = a_m.merge(
    b_m[['account_id', 'transaction_datetime_b', 'transaction_amount_b']],
    on='account_id',
    how='left'
)

mask_1m = (
    (merged_m['transaction_datetime_b'] >= merged_m['transaction_datetime_a'] - pd.DateOffset(months=1)) &
    (merged_m['transaction_datetime_b'] <= merged_m['transaction_datetime_a'])
)

step2_monthly = (
    merged_m.loc[mask_1m, [
        'transaction_id',
        'account_id',
        'customer_id',
        'transaction_datetime_a',
        'transaction_amount_a',
        'transaction_amount_b',
        'transaction_datetime_b'
    ]]
    .rename(columns={
        'transaction_datetime_a': 'transaction_datetime',
        'transaction_amount_a': 'transaction_amount',
        'transaction_amount_b': 'amount_v1',
        'transaction_datetime_b': 'trxn_date_v1'
    })
    .sort_values(['account_id', 'transaction_datetime', 'trxn_date_v1'])
)

print("\nSTEP 2 (monthly) - row count:", len(step2_monthly))
print("\nSTEP 2 (monthly) - sample:")
print(step2_monthly.head(50))

# STEP 3 – aggregation: group by account_id and month (EOMONTH)
step3_monthly = (
    step2_monthly
    .assign(
        month_last_date = step2_monthly['transaction_datetime'].dt.to_period('M').dt.to_timestamp('M')
    )
    .groupby(['account_id', 'month_last_date'], as_index=False)
    .agg(sum_amount_v1=('amount_v1', 'sum'))
)

print("\nSTEP 3 (monthly) - row count:", len(step3_monthly))
print("\nSTEP 3 (monthly) - sample:")
print(step3_monthly.head(50))

ValueError: time data "21/12/2024" doesn't match format "%m/%d/%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
import numpy as np
import pandas as pd

# ==============================================================
# STEP 3.0 - CONFIG (Aggregation Lens + Boundary Strategy)
# ==============================================================
ENTITY_COLLAPSE = 'max'      # 'max', 'avg', 'p95', 'last'
TIME_LENS = 'full'           # 'full', 'daily', 'rolling_peak', 'sustained'
SUSTAINED_DAYS = 3           # used only if TIME_LENS in ('rolling_peak', 'sustained')

THRESHOLD_STRATEGY = 'percentile'   # 'percentile', 'absolute', 'top_n'
PERCENTILE = 99.0
ABSOLUTE_THRESHOLD = None    # used if strategy == 'absolute'
TOP_N = 100                  # used if strategy == 'top_n'

BUFFER_TYPE = 'hard'         # 'hard' or 'buffered'
BUFFER_BAND_PCT = 2.0        # only if buffered

STRESS_DELTAS_PCT = [-5, -2, -1, 1, 2, 5]

STABILITY_SAMPLES = 20
STABILITY_SAMPLE_FRAC = 0.75

# ==============================================================
# STEP 3.1 - Build Behavior Table from Step 2 (threshold_table)
# ==============================================================
# threshold_table is expected from your Step 2 script
# Columns expected: account_id, transaction_datetime, threshold_amt
if 'threshold_table' not in globals():
    raise ValueError("threshold_table not found. Run Step 2 first.")

behavior_df = threshold_table.rename(columns={
    'account_id': 'entity_id',
    'transaction_datetime': 'as_of_date',
    'threshold_amt': 'metric_value'
}).copy()

behavior_df['as_of_date'] = pd.to_datetime(behavior_df['as_of_date'], errors='coerce')
behavior_df['metric_value'] = pd.to_numeric(behavior_df['metric_value'], errors='coerce')
behavior_df = behavior_df.dropna(subset=['as_of_date', 'metric_value'])

# --------------------------------------------------------------
# Helpers
# --------------------------------------------------------------
def apply_time_lens(df, time_lens='full', sustained_days=3):
    time_lens = (time_lens or 'full').lower()
    base = df[['entity_id', 'as_of_date', 'metric_value']].copy()

    if time_lens == 'full':
        return base

    # daily collapse
    daily = (
        base.assign(day=base['as_of_date'].dt.floor('D'))
            .groupby(['entity_id', 'day'], as_index=False)['metric_value']
            .max()
            .rename(columns={'day': 'as_of_date'})
            .sort_values(['entity_id', 'as_of_date'])
    )

    if time_lens == 'daily':
        return daily

    # rolling_peak or sustained
    def roll_group(g):
        g = g.sort_values('as_of_date')
        vals = g['metric_value']
        roll_sum = vals.rolling(sustained_days, min_periods=1).sum()
        roll_avg = vals.rolling(sustained_days, min_periods=1).mean()
        all_non_zero = vals.rolling(sustained_days, min_periods=1).apply(lambda x: 1 if np.all(x != 0) else 0)
        if time_lens == 'rolling_peak':
            g['metric_value'] = roll_sum.values
        elif time_lens == 'sustained':
            g['metric_value'] = np.where(all_non_zero.values == 1, roll_avg.values, np.nan)
        return g

    rolled = daily.groupby('entity_id', as_index=False, group_keys=False).apply(roll_group)
    rolled = rolled.dropna(subset=['metric_value'])
    return rolled

def collapse_entity(df, entity_collapse='max'):
    entity_collapse = (entity_collapse or 'max').lower()
    g = df.groupby('entity_id')['metric_value']
    if entity_collapse == 'max':
        return g.max().reset_index(name='aggregated_value')
    if entity_collapse == 'avg':
        return g.mean().reset_index(name='aggregated_value')
    if entity_collapse == 'p95':
        return g.quantile(0.95).reset_index(name='aggregated_value')
    if entity_collapse == 'last':
        idx = df.sort_values('as_of_date').groupby('entity_id').tail(1)
        return idx[['entity_id', 'metric_value']].rename(columns={'metric_value': 'aggregated_value'})
    raise ValueError("Unsupported entity_collapse")

def ks_stat(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return None
    data = np.concatenate([a, b])
    a_sorted = np.sort(a)
    b_sorted = np.sort(b)
    cdf_a = np.searchsorted(a_sorted, data, side='right') / len(a_sorted)
    cdf_b = np.searchsorted(b_sorted, data, side='right') / len(b_sorted)
    return float(np.max(np.abs(cdf_a - cdf_b)))

def j_curve(signal, labels, points=120):
    signal = np.asarray(signal, dtype=float)
    labels = np.asarray(labels, dtype=int)
    mask = np.isfinite(signal)
    signal = signal[mask]
    labels = labels[mask]
    if len(signal) == 0:
        return {'max_j': None, 'threshold': None}

    thresholds = np.quantile(signal, np.linspace(0, 1, points))
    pos = labels == 1
    neg = labels == 0
    if pos.sum() == 0 or neg.sum() == 0:
        return {'max_j': None, 'threshold': None}

    best_j = -1
    best_th = None
    for th in thresholds:
        tpr = (signal[pos] >= th).mean()
        fpr = (signal[neg] >= th).mean()
        j = tpr - fpr
        if j > best_j:
            best_j = j
            best_th = th
    return {'max_j': float(best_j), 'threshold': float(best_th)}

# ==============================================================
# STEP 3.1 - Population validated (aggregation lens applied)
# ==============================================================
lens_df = apply_time_lens(behavior_df, TIME_LENS, SUSTAINED_DAYS)
agg_df = collapse_entity(lens_df, ENTITY_COLLAPSE)

step3_1_summary = {
    'n_entities': int(len(agg_df)),
    'min': float(agg_df['aggregated_value'].min()),
    'max': float(agg_df['aggregated_value'].max()),
    'mean': float(agg_df['aggregated_value'].mean()),
    'median': float(agg_df['aggregated_value'].median())
}

print("STEP 3.1 - Population summary:", step3_1_summary)

# ==============================================================
# STEP 3.2 - Aggregation lens recorded
# ==============================================================
step3_2_lens = {
    'entity_collapse': ENTITY_COLLAPSE,
    'time_lens': TIME_LENS,
    'sustained_days': SUSTAINED_DAYS
}
print("STEP 3.2 - Aggregation lens:", step3_2_lens)

# ==============================================================
# STEP 3.3 - Boundary construction (threshold strategy)
# ==============================================================
values = agg_df['aggregated_value'].values
if THRESHOLD_STRATEGY == 'percentile':
    threshold_value = float(np.percentile(values, PERCENTILE))
elif THRESHOLD_STRATEGY == 'absolute':
    threshold_value = float(ABSOLUTE_THRESHOLD)
elif THRESHOLD_STRATEGY == 'top_n':
    threshold_value = float(np.sort(values)[-TOP_N]) if len(values) >= TOP_N else float(values.max())
else:
    raise ValueError("Unsupported strategy")

if BUFFER_TYPE == 'buffered':
    lower = threshold_value * (1 - BUFFER_BAND_PCT / 100.0)
    upper = threshold_value * (1 + BUFFER_BAND_PCT / 100.0)
else:
    lower = threshold_value
    upper = threshold_value

atl_entities = set(agg_df[agg_df['aggregated_value'] >= upper]['entity_id'])
btl_entities = set(agg_df[agg_df['aggregated_value'] < lower]['entity_id'])

step3_3_boundary = {
    'threshold_value': threshold_value,
    'lower': lower,
    'upper': upper,
    'atl_count': len(atl_entities),
    'btl_count': len(btl_entities)
}
print("STEP 3.3 - Boundary:", step3_3_boundary)

# ==============================================================
# STEP 3.4 - KS validation
# ==============================================================
label_map = agg_df.set_index('entity_id')['aggregated_value']
behavior_labeled = behavior_df.copy()
behavior_labeled['aggregated_value'] = behavior_labeled['entity_id'].map(label_map)

behavior_labeled['pop'] = np.where(
    behavior_labeled['aggregated_value'] >= upper, 'ATL',
    np.where(behavior_labeled['aggregated_value'] < lower, 'BTL', 'EXCLUDE')
)
atl_vals = behavior_labeled.loc[behavior_labeled['pop'] == 'ATL', 'metric_value'].values
btl_vals = behavior_labeled.loc[behavior_labeled['pop'] == 'BTL', 'metric_value'].values

step3_4_ks = {
    'ks_stat': ks_stat(atl_vals, btl_vals),
    'n_atl': int(len(atl_vals)),
    'n_btl': int(len(btl_vals))
}
print("STEP 3.4 - KS:", step3_4_ks)

# ==============================================================
# STEP 3.5 - Stress testing
# ==============================================================
base_ids = atl_entities
base_atl_count = max(1, len(base_ids))
base_volume = float(agg_df.loc[agg_df['entity_id'].isin(base_ids), 'aggregated_value'].abs().sum())

stress_results = []
for d in STRESS_DELTAS_PCT:
    th = upper * (1 + d / 100.0)
    ids = set(agg_df[agg_df['aggregated_value'] >= th]['entity_id'])
    enter = len(ids - base_ids)
    leave = len(base_ids - ids)
    churn_pct = (enter + leave) / base_atl_count * 100.0
    enter_pct = enter / base_atl_count * 100.0
    leave_pct = leave / base_atl_count * 100.0
    sym_ids = list(ids.symmetric_difference(base_ids))
    volume_churn = float(agg_df.loc[agg_df['entity_id'].isin(sym_ids), 'aggregated_value'].abs().sum())
    volume_churn_pct = (volume_churn / base_volume * 100.0) if base_volume else 0.0
    stress_results.append({
        'delta_pct': d,
        'entity_churn_pct': churn_pct,
        'volume_churn_pct': volume_churn_pct,
        'enter_pct': enter_pct,
        'leave_pct': leave_pct
    })

step3_5_stress = stress_results
print("STEP 3.5 - Stress results (first 3 rows):", step3_5_stress[:3])

# ==============================================================
# STEP 3.6 - J-statistic separation + stability
# ==============================================================
labels = np.where(behavior_labeled['pop'] == 'ATL', 1, np.where(behavior_labeled['pop'] == 'BTL', 0, -1))
mask = labels >= 0
signal = behavior_labeled.loc[mask, 'metric_value'].values
labels = labels[mask]

j_result = j_curve(signal, labels)

# Stability via bootstrap
j_samples = []
if len(signal) > 0:
    n = len(signal)
    sample_n = max(10, int(n * STABILITY_SAMPLE_FRAC))
    rng = np.random.default_rng(42)
    for _ in range(STABILITY_SAMPLES):
        idx = rng.choice(n, size=sample_n, replace=False)
        js = j_curve(signal[idx], labels[idx])
        if js['max_j'] is not None:
            j_samples.append(js['max_j'])

if j_samples:
    mean_j = float(np.mean(j_samples))
    std_j = float(np.std(j_samples))
    if std_j <= 0.02:
        stability_label = 'stable'
    elif std_j <= 0.05:
        stability_label = 'moderate'
    else:
        stability_label = 'fragile'
else:
    mean_j = None
    std_j = None
    stability_label = 'unknown'

step3_6_j = {
    'max_j': j_result['max_j'],
    'j_threshold': j_result['threshold'],
    'stability_mean_j': mean_j,
    'stability_std_j': std_j,
    'stability_label': stability_label
}
print("STEP 3.6 - J statistic:", step3_6_j)
